In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)




In [ ]:
import pandas as pd
import json

# Load JSON
with open("../data/train_logical_combinations_output.json", "r") as f:
    data = json.load(f)

# Convert the 'questions' list to a DataFrame
train_df = pd.DataFrame(data["questions"])
train_df.to_csv('train_logical_combinations_output.csv',index=False)

train_df.head()

# Load JSON
with open("../data/dev_logical_combinations_output.json", "r") as f:
    data = json.load(f)

# Convert the 'questions' list to a DataFrame
dev_df = pd.DataFrame(data["questions"])
dev_df.to_csv('dev_logical_combinations_output.csv',index=False)

dev_df.head()


# Load JSON
with open("../data/test_logical_combinations_output.json", "r") as f:
    data = json.load(f)

# Convert the 'questions' list to a DataFrame
test_df = pd.DataFrame(data["questions"])
test_df.to_csv('test_logical_combinations_output.csv',index=False)

test_df.head()

In [ ]:
train_df.head()

In [ ]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch
from torch.optim import AdamW
from tqdm.auto import tqdm
import json
from collections import defaultdict
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt 

import random
import os
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


# Load model
model_name = "microsoft/deberta-v3-base"

training_config = {
    'max_epochs': 10,
    'patience': 3,  # Early stopping patience
    'min_delta': 0.001,  # Minimum improvement to count
    'batch_size': 4,
    'learning_rate': 2e-5,
    'warmup_steps': 500,
    'gradient_accumulation_steps': 2,
    'weight_decay': 0.01,
    'max_length': 256
}

print("\nTraining Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

# Custom Dataset
class MCQADataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length=256):
        self.data = []
        with open(json_path, 'r') as f:
            for line in f:
                if line.strip():
                    self.data.append(json.loads(line.strip()))
        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"Loaded {len(self.data)} examples from {json_path}")
        
        # Print dataset statistics
        type_counts = defaultdict(int)
        for item in self.data:
            type_counts[item.get("qa_type", "unknown")] += 1
        print("Dataset composition:")
        for qa_type, count in sorted(type_counts.items()):
            print(f"  {qa_type}: {count} ({count/len(self.data)*100:.1f}%)")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        choices = item["choices"]
        label = item["label"]
        qa_type = item.get("qa_type", "unknown")
        
        tokenized = self.tokenizer(
            [question] * 4,
            choices,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": torch.tensor(label),
            "qa_type": qa_type
        }

def compute_metrics_by_type(predictions, labels, qa_types):
    """
    Compute comprehensive metrics for dataset paper
    Returns per-type metrics with BOTH micro and macro averaging
    """
    predictions = np.array(predictions)
    labels = np.array(labels)
    qa_types = np.array(qa_types)
    
    unique_types = sorted([t for t in set(qa_types) if t != 'overall'])
    
    results = {
        'per_type': {},
        'macro_across_types': {},
        'micro_overall': {},
        'confusion_matrices': {}
    }
    
    # Per-type metrics (with both micro and macro within type)
    for qa_type in unique_types:
        mask = qa_types == qa_type
        type_preds = predictions[mask]
        type_labels = labels[mask]
        
        if len(type_preds) == 0:
            continue
        
        # MACRO: Average metrics across the 4 answer choices for this type
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
            type_labels, type_preds, average='macro', zero_division=0, labels=[0, 1, 2, 3]
        )
        
        # MICRO: Global metrics for this type (same as accuracy)
        micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
            type_labels, type_preds, average='micro', zero_division=0, labels=[0, 1, 2, 3]
        )
        
        # Per-class metrics for this type (for detailed analysis)
        per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
            type_labels, type_preds, average=None, zero_division=0, labels=[0, 1, 2, 3]
        )
        
        accuracy = (type_preds == type_labels).mean()
        
        results['per_type'][qa_type] = {
            # Macro metrics (average across 4 answer choices)
            'macro': {
                'precision': macro_precision,
                'recall': macro_recall,
                'f1': macro_f1,
            },
            # Micro metrics (global for this type)
            'micro': {
                'precision': micro_precision,  # same as accuracy
                'recall': micro_recall,        # same as accuracy
                'f1': micro_f1,                # same as accuracy
                'accuracy': accuracy,
            },
            # Per-answer-choice metrics
            'per_choice': {
                f'choice_{i}': {
                    'precision': per_class_precision[i],
                    'recall': per_class_recall[i],
                    'f1': per_class_f1[i],
                    'support': per_class_support[i]
                } for i in range(4)
            },
            'support': len(type_preds),
            'correct': (type_preds == type_labels).sum()
        }
        
        # Confusion matrix for this type
        results['confusion_matrices'][qa_type] = confusion_matrix(
            type_labels, type_preds, labels=[0, 1, 2, 3]
        )
    
    # Macro metrics across logical types (what you typically report in paper)
    macro_across_types_precision = np.mean([results['per_type'][t]['macro']['precision'] for t in unique_types])
    macro_across_types_recall = np.mean([results['per_type'][t]['macro']['recall'] for t in unique_types])
    macro_across_types_f1 = np.mean([results['per_type'][t]['macro']['f1'] for t in unique_types])
    macro_across_types_accuracy = np.mean([results['per_type'][t]['micro']['accuracy'] for t in unique_types])
    
    results['macro_across_types'] = {
        'precision': macro_across_types_precision,
        'recall': macro_across_types_recall,
        'f1': macro_across_types_f1,
        'accuracy': macro_across_types_accuracy
    }
    
    # Micro metrics overall (global across all instances)
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average='micro', zero_division=0, labels=[0, 1, 2, 3]
    )
    micro_accuracy = (predictions == labels).mean()
    
    results['micro_overall'] = {
        'accuracy': micro_accuracy,
        'precision': micro_precision,
        'recall': micro_recall,
        'f1': micro_f1,
        'support': len(predictions),
        'correct': (predictions == labels).sum()
    }
    
    return results

def evaluate_comprehensive(model, dataloader, device):
    """Comprehensive evaluation with all metrics"""
    model.eval()
    
    all_predictions = []
    all_labels = []
    all_types = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            qa_types = batch["qa_type"]
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = outputs.logits.argmax(dim=-1)
            
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_types.extend(qa_types)
    
    metrics = compute_metrics_by_type(all_predictions, all_labels, all_types)
    return metrics

def print_results_table(metrics, epoch=None, model_name=None):
    """Print results table"""
    
    if epoch is not None:
        if model_name:
            print(f"Model: {model_name} | Epoch {epoch}")
    
    print(f"\n{'Type':<12} {'Accuracy':<12} {'Macro-P':<12} {'Macro-R':<12} {'Macro-F1':<12} {'Support':<10}")
    
    for qa_type in sorted(metrics['per_type'].keys()):
        m_macro = metrics['per_type'][qa_type]['macro']
        m_micro = metrics['per_type'][qa_type]['micro']
        support = metrics['per_type'][qa_type]['support']
        
        print(f"{qa_type:<12} {m_micro['accuracy']:>10.4f} {m_macro['precision']:>10.4f} "
              f"{m_macro['recall']:>10.4f} {m_macro['f1']:>10.4f} {support:>8}")
    
    
    m = metrics['macro_across_types']
    print(f"{'Macro Avg':<12} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "
          f"{m['recall']:>10.4f} {m['f1']:>10.4f} {'-':>8}")
    
    m = metrics['micro_overall']
    print(f"{'Micro Avg':<12} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "
          f"{m['recall']:>10.4f} {m['f1']:>10.4f} {m['support']:>8}")
    

# Load datasets
train_dataset = MCQADataset("../data/train_all_hf.json", 
                           tokenizer, max_length=256)
eval_dataset = MCQADataset("../data/dev_all_hf.json", 
                          tokenizer, max_length=256)

# Setup data loaders
batch_size = training_config['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size)

# Setup model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
model.to(device)

# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), 
                 lr=training_config['learning_rate'], 
                 weight_decay=training_config['weight_decay'])
num_training_steps = training_config['max_epochs'] * len(train_loader) // training_config['gradient_accumulation_steps']
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=training_config['warmup_steps'],
    num_training_steps=num_training_steps
)
# Training loop with early stopping
best_macro_f1 = 0
best_epoch = 0
epochs_without_improvement = 0
best_metrics = None
training_history = []

print(f"Training {model_name} with Early Stopping")
print(f"Max Epochs: {training_config['max_epochs']}, Patience: {training_config['patience']}")

for epoch in range(training_config['max_epochs']):
    print(f"Epoch {epoch + 1}/{training_config['max_epochs']}")
    
    # Training
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc="Training")
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass (no mixed precision for P100)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss / training_config['gradient_accumulation_steps']
        
        loss.backward()
        
        # Gradient accumulation
        if (step + 1) % training_config['gradient_accumulation_steps'] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * training_config['gradient_accumulation_steps']
        progress_bar.set_postfix({"loss": f"{total_loss / (step + 1):.4f}"})
    
    avg_train_loss = total_loss / len(train_loader)
    print(f"\nTrain Loss: {avg_train_loss:.4f}")
    
    # Evaluation
    print("\nEvaluating...")
    metrics = evaluate_comprehensive(model, eval_loader, device)
    print_results_table(metrics, epoch=epoch + 1, model_name=model_name)
    
    macro_f1 = metrics['macro_across_types']['f1']
    macro_acc = metrics['macro_across_types']['accuracy']
    
    # Save history
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'macro_f1': macro_f1,
        'macro_accuracy': macro_acc,
        'macro_precision': metrics['macro_across_types']['precision'],
        'macro_recall': metrics['macro_across_types']['recall']
    })
    
    # Early stopping check
    improvement = macro_f1 - best_macro_f1
    
    if improvement > training_config['min_delta']:
        best_macro_f1 = macro_f1
        best_epoch = epoch + 1
        best_metrics = metrics
        epochs_without_improvement = 0
        
        save_dir = f"./{model_name.replace('/', '-')}-best"
        print(f"\n  New best Macro F1: {macro_f1:.4f} (+{improvement:.4f})! Saving to {save_dir}")
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        
        def convert_to_serializable(obj):
            if isinstance(obj, dict):
                return {k: convert_to_serializable(v) for k, v in obj.items()}
            elif isinstance(obj, (np.integer, np.floating)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            return obj
        
        results_for_paper = {
            'model': model_name,
            'best_epoch': best_epoch,
            'training_config': training_config,
            'training_history': training_history,
            'best_metrics': convert_to_serializable(best_metrics)
        }
        
        with open(f"{save_dir}/results_detailed.json", "w") as f:
            json.dump(results_for_paper, f, indent=2)
    else:
        epochs_without_improvement += 1
        print(f"\n   No improvement for {epochs_without_improvement} epoch(s) "
              f"(current: {macro_f1:.4f}, best: {best_macro_f1:.4f})")
    
    # Early stopping
    if epochs_without_improvement >= training_config['patience']:
        print(f"Early stopping triggered!")
        print(f"Best Macro F1: {best_macro_f1:.4f} at epoch {best_epoch}")
        print(f"Training stopped at epoch {epoch + 1}")
        break

# Final summary
print("Training Complete!")
print(f"Model: {model_name}")
print(f"Best Epoch: {best_epoch}/{epoch + 1}")
print(f"Best Macro F1: {best_macro_f1:.4f}")

if best_metrics is not None:
    print(f"Best Validation Macro F1: {best_macro_f1:.4f}")
    print(f"Best Validation Macro Accuracy: {best_metrics['macro_across_types']['accuracy']:.4f}")
else:
    print("Best Validation Macro F1: n/a (no improvement over initial baseline)")
    print("Best Validation Macro Accuracy: n/a")


# Plot training curves
plt.figure(figsize=(15, 5))

epochs_list = [h['epoch'] for h in training_history]

plt.subplot(1, 3, 1)
plt.plot(epochs_list, [h['train_loss'] for h in training_history], marker='o', label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs_list, [h['macro_f1'] for h in training_history], marker='o', color='green', label='Macro F1')
plt.axvline(x=best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Macro F1')
plt.title('Validation Macro F1')
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs_list, [h['macro_accuracy'] for h in training_history], marker='o', color='blue', label='Macro Accuracy')
plt.axvline(x=best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Macro Accuracy')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(f'{model_name.replace("/", "-")}_training_curves.png', dpi=300, bbox_inches='tight')
print(f"\n  Saved training curves to {model_name.replace('/', '-')}_training_curves.png")

# Summary table for paper
print("SUMMARY FOR PAPER:")
print(f"Model: {model_name}")
print(f"Best Epoch: {best_epoch}")
print(f"Total Epochs Trained: {len(training_history)}")
print(f"Converged: {'Yes (early stopping)' if epochs_without_improvement >= training_config['patience'] else 'No (completed all epochs)'}")
print(f"Best Validation Macro F1: {best_macro_f1:.4f}")
print(f"Best Validation Macro Accuracy: {best_metrics['macro_across_types']['accuracy']:.4f}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch
from torch.optim import AdamW
from tqdm.auto import tqdm
import json
from collections import defaultdict
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Load model
model_path = "/microsoft-deberta-v3-base-best" #change this to microsoft/deberta-v3-base for baseline results
model_name = 'deberta-v3'


eval_config = {
    'batch_size': 4,  # Can use larger batch for inference
    'max_length': 256
}

print("\nEvaluation Configuration:")
for key, value in eval_config.items():
    print(f"  {key}: {value}")
print(f"Model: {model_name}")

print(f"\nLoading model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForMultipleChoice.from_pretrained(model_path)

class MCQADataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length=256, start_idx=None, end_idx=None):
        self.data = []
        with open(json_path, 'r') as f:
            for line in f:
                if line.strip():
                    self.data.append(json.loads(line.strip()))

        # Apply partitioning
        if start_idx is not None or end_idx is not None:
            self.data = self.data[start_idx:end_idx]

        self.tokenizer = tokenizer
        self.max_length = max_length

        print(f"Loaded {len(self.data)} examples from {json_path}")        
        # Print dataset statistics
        type_counts = defaultdict(int)
        for item in self.data:
            type_counts[item.get("qa_type", "unknown")] += 1
        print("Dataset composition:")
        for qa_type, count in sorted(type_counts.items()):
            print(f"  {qa_type}: {count} ({count/len(self.data)*100:.1f}%)")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        choices = item["choices"]
        label = item["label"]
        qa_type = item.get("qa_type", "unknown")
        
        tokenized = self.tokenizer(
            [question] * 4,
            choices,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": torch.tensor(label),
            "qa_type": qa_type
        }


def compute_metrics_by_type(predictions, labels, qa_types):
    """Compute comprehensive metrics"""
    predictions = np.array(predictions)
    labels = np.array(labels)
    qa_types = np.array(qa_types)
    
    unique_types = sorted([t for t in set(qa_types) if t != 'overall'])
    
    results = {
        'per_type': {},
        'macro_across_types': {},
        'micro_overall': {},
        'confusion_matrices': {}
    }
    
    for qa_type in unique_types:
        mask = qa_types == qa_type
        type_preds = predictions[mask]
        type_labels = labels[mask]
        
        if len(type_preds) == 0:
            continue
        
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
            type_labels, type_preds, average='macro', zero_division=0, labels=[0, 1, 2, 3]
        )
        
        micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
            type_labels, type_preds, average='micro', zero_division=0, labels=[0, 1, 2, 3]
        )
        
        per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
            type_labels, type_preds, average=None, zero_division=0, labels=[0, 1, 2, 3]
        )
        
        accuracy = (type_preds == type_labels).mean()
        
        results['per_type'][qa_type] = {
            'macro': {
                'precision': macro_precision,
                'recall': macro_recall,
                'f1': macro_f1,
            },
            'micro': {
                'precision': micro_precision,
                'recall': micro_recall,
                'f1': micro_f1,
                'accuracy': accuracy,
            },
            'per_choice': {
                f'choice_{i}': {
                    'precision': per_class_precision[i],
                    'recall': per_class_recall[i],
                    'f1': per_class_f1[i],
                    'support': per_class_support[i]
                } for i in range(4)
            },
            'support': len(type_preds),
            'correct': (type_preds == type_labels).sum()
        }
        
        results['confusion_matrices'][qa_type] = confusion_matrix(
            type_labels, type_preds, labels=[0, 1, 2, 3]
        )
    
    macro_across_types_precision = np.mean([results['per_type'][t]['macro']['precision'] for t in unique_types])
    macro_across_types_recall = np.mean([results['per_type'][t]['macro']['recall'] for t in unique_types])
    macro_across_types_f1 = np.mean([results['per_type'][t]['macro']['f1'] for t in unique_types])
    macro_across_types_accuracy = np.mean([results['per_type'][t]['micro']['accuracy'] for t in unique_types])
    
    results['macro_across_types'] = {
        'precision': macro_across_types_precision,
        'recall': macro_across_types_recall,
        'f1': macro_across_types_f1,
        'accuracy': macro_across_types_accuracy
    }
    
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average='micro', zero_division=0, labels=[0, 1, 2, 3]
    )
    micro_accuracy = (predictions == labels).mean()
    
    results['micro_overall'] = {
        'accuracy': micro_accuracy,
        'precision': micro_precision,
        'recall': micro_recall,
        'f1': micro_f1,
        'support': len(predictions),
        'correct': (predictions == labels).sum()
    }
    
    return results

def evaluate_zero_shot(model, dataloader, tokenizer, device):
    """Zero-shot evaluation"""
    model.eval()
    
    all_predictions = []
    all_labels = []
    all_types = []
    
    print("\nRunning zero-shot evaluation...")
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            correct_labels = batch["labels"]
            qa_types = batch["qa_type"]
            
            # Generate predictions
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = outputs.logits.argmax(dim=-1)
            
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(correct_labels.cpu().tolist())
            all_types.extend(qa_types)
    
    metrics = compute_metrics_by_type(all_predictions, all_labels, all_types)
    return metrics


def print_results_table(metrics, model_name=None):
    """Print results table"""
    
    if model_name:
        print(f"Model: {model_name} (Zero-Shot)")
    
    print(f"\n{'Type':<12} {'Accuracy':<12} {'Macro-P':<12} {'Macro-R':<12} {'Macro-F1':<12} {'Support':<10}")
    
    for qa_type in sorted(metrics['per_type'].keys()):
        m_macro = metrics['per_type'][qa_type]['macro']
        m_micro = metrics['per_type'][qa_type]['micro']
        support = metrics['per_type'][qa_type]['support']
        
        print(f"{qa_type:<12} {m_micro['accuracy']:>10.4f} {m_macro['precision']:>10.4f} "
              f"{m_macro['recall']:>10.4f} {m_macro['f1']:>10.4f} {support:>8}")
    
    
    m = metrics['macro_across_types']
    print(f"{'Macro Avg':<12} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "
          f"{m['recall']:>10.4f} {m['f1']:>10.4f} {'-':>8}")
    
    m = metrics['micro_overall']
    print(f"{'Micro Avg':<12} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "
          f"{m['recall']:>10.4f} {m['f1']:>10.4f} {m['support']:>8}")
    


# MAIN EVALUATION

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model.to(device)

# Load test dataset
dev_dataset = MCQADataset(
    "../data/dev_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length']
)

# Setup data loader
dev_loader = DataLoader(dev_dataset, batch_size=eval_config['batch_size'])

#AND
test_dataset_first = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=0,
    end_idx=250
)

# Setup data loader
test_loader_first = DataLoader(test_dataset_first, batch_size=eval_config['batch_size'])

test_metrics = evaluate_zero_shot(model, test_loader_first, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

test_dataset_second = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=250,
    end_idx=500
)

# Setup data loader
test_loader_second = DataLoader(test_dataset_second, batch_size=eval_config['batch_size'])


test_metrics = evaluate_zero_shot(model, test_loader_second, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

#OR
test_dataset_first = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=500,
    end_idx=750
)


# Setup data loader
test_loader_first = DataLoader(test_dataset_first, batch_size=eval_config['batch_size'])

test_metrics = evaluate_zero_shot(model, test_loader_first, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

test_dataset_second = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=750,
    end_idx=1000
)

# Setup data loader
test_loader_second = DataLoader(test_dataset_second, batch_size=eval_config['batch_size'])


test_metrics = evaluate_zero_shot(model, test_loader_second, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

#NNOR
test_dataset_first = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=1000,
    end_idx=1250
)


# Setup data loader
test_loader_first = DataLoader(test_dataset_first, batch_size=eval_config['batch_size'])

test_metrics = evaluate_zero_shot(model, test_loader_first, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

test_dataset_second = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=1250,
    end_idx=1500
)

# Setup data loader
test_loader_second = DataLoader(test_dataset_second, batch_size=eval_config['batch_size'])


test_metrics = evaluate_zero_shot(model, test_loader_second, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

#Mixed
test_dataset_first = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=1500,
    end_idx=1750
)


# Setup data loader
test_loader_first = DataLoader(test_dataset_first, batch_size=eval_config['batch_size'])

test_metrics = evaluate_zero_shot(model, test_loader_first, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)

test_dataset_second = MCQADataset(
    "../data/test_all_hf.json",  # or test file
    tokenizer, 
    max_length=eval_config['max_length'],
    start_idx=1750,
    end_idx=2000
)

# Setup data loader
test_loader_second = DataLoader(test_dataset_second, batch_size=eval_config['batch_size'])


test_metrics = evaluate_zero_shot(model, test_loader_second, tokenizer, device)

# Print results
print_results_table(test_metrics, model_name=model_name)


# Run zero-shot evaluation
print(f"Zero-Shot Dev Set Evaluation: {model_name}")

dev_metrics = evaluate_zero_shot(model, dev_loader, tokenizer, device)

# Print results
print_results_table(dev_metrics, model_name=model_name)

# Run zero-shot evaluation
print(f"Zero-Shot Test Set Evaluation: {model_name}")



# Save results
def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

# Save both
results_for_paper = {
    'model': model_name,
    'evaluation_type': 'zero-shot',
    'config': eval_config,
    'dev_metrics': convert_to_serializable(dev_metrics),
    'test_metrics': convert_to_serializable(test_metrics),
}

output_file = f"{model_name.replace('/', '-')}_zeroshot_results.json"
with open(output_file, "w") as f:
    json.dump(results_for_paper, f, indent=2)

print(f"\nResults saved to {output_file}")

# Summary based on test set
print("ZERO-SHOT RESULTS SUMMARY (TEST SET):")
print(f"Model: {model_name}")
print(f"Evaluation Type: Zero-Shot (No Training)")
print(f"Test Examples: {test_metrics['micro_overall']['support']}")
print(f"Macro F1 (across types): {test_metrics['macro_across_types']['f1']:.4f}")
print(f"Macro Accuracy (across types): {test_metrics['macro_across_types']['accuracy']:.4f}")
print(f"Micro F1 (overall): {test_metrics['micro_overall']['f1']:.4f}")
print(f"Micro Accuracy (overall): {test_metrics['micro_overall']['accuracy']:.4f}")